In [27]:
import pandas as pd 
import numpy as np

In [28]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [29]:
df = pd.read_csv("covid_toy.csv")

In [30]:
df.head()


,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [31]:
 df['cough'].value_counts()

cough
Mild      62
Strong    38
Name: count, dtype: int64

In [32]:
df['city'].value_counts()

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [33]:
df.isnull().sum()  # To find null values from all data

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [34]:
from sklearn.model_selection import train_test_split
X_train,X_test , y_train , y_test = train_test_split(df.drop(columns = ['has_covid']),df['has_covid'],
                                                     test_size = 0.2)

In [35]:
X_train

,age,gender,fever,cough,city
80,14,Female,99.0,Mild,Mumbai
29,34,Female,NaN,Strong,Mumbai
73,34,Male,98.0,Strong,Kolkata
71,75,Female,104.0,Strong,Delhi
33,26,Female,98.0,Mild,Kolkata
...,...,...,...,...,...
8,19,Female,100.0,Strong,Bangalore
18,64,Female,98.0,Mild,Bangalore
88,5,Female,100.0,Mild,Kolkata
58,23,Male,98.0,Strong,Mumbai


# 1. AAM JINDAGI

## without using colm transformer

In [44]:
# Adding Simple Imputer to fever Col


si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])   # 80

# Also the test data
X_test_fever = si.transform(X_test[['fever']])       # 20  
# No need to use fit_transform for Test Data , we have to use only transform, coz in trainning data alredy fited and learned
'''fit()
Function : Learns the necessary parameters (e.g., mean and standard deviation for scaling, or model weights for training) from the provided data but does not apply the change.

Use case : Used on the training data to learn parameters that will then be applied consistently to both training and test sets.

fit_transform()
Function : Combines fit() and transform() into one step. It learns the parameters and transforms the data simultaneously.

Use Case : Used on the training data for convenience and efficiency in data preprocessing pipelines.'''



X_test_fever.shape

(20, 1)

In [37]:
# Ordinal encoding  --> Cough

oe = OrdinalEncoder(categories=[['Mild' , 'Strong']])   
# Mild --> small value # Strong --> Big Value
X_train_cough = oe.fit_transform(X_train[['cough']])

X_test_cough  = oe.transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [50]:
# OneHotEncoding  --> Gender, City

ohe = OneHotEncoder(drop = 'first' , sparse_output=False )  
# , sparse = False  --> it is must required step , because other wise itwill produce o/p in sparse and we can't concatenate it

X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])

X_test_gender_city = ohe.transform(X_test[['gender','city']])

X_test_gender_city.shape
#X_train_gender_city

(20, 4)

In [51]:
# Extracting Age --> Becoz we didn't work on it

X_train_age = X_train.drop(columns = ['gender',	'fever',	'cough',	'city']).values

X_test_age = X_test.drop(columns = ['gender',	'fever',	'cough',	'city']).values

X_train_age.shape   # 1 column --> age

(80, 1)

In [52]:
## Concatinating all the Columns


X_train_transformed = np.concatenate((X_train_age, X_train_fever, X_train_gender_city, X_train_cough), axis=1)

X_test_transformed = np.concatenate((X_test_age, X_test_fever, X_test_gender_city, X_test_cough), axis=1)
X_train_transformed.shape

(80, 7)

# Mentos Zindagi


## one line solution for Tranform Column's

In [53]:
from sklearn.compose import ColumnTransformer

In [56]:
transformer = ColumnTransformer(transformers = [
    ('tnf1', SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild' , 'Strong']]),['cough']),
    ('tnf3',OneHotEncoder(drop = 'first' , sparse_output=False ) , ['gender','city']  )
], remainder = 'passthrough')

In [58]:
transformer.fit_transform(X_train).shape

(80, 7)